# Phase 2: Deep Learning, Ablations and a Hybrid Model
**Project 11: Inventory Image Tagging via Handcrafted Features and a Small CNN (Fashion-MNIST)**

This notebook runs the whole Phase 2 study and produces every table and figure of the Phase 2 report.

| Step | What it does | Data used | Time on a T4 GPU |
|---|---|---|---|
| 1 | Setup, unit tests, 2-minute smoke run | tiny subsets | about 5 min |
| 2 | Train 16 configurations x 3 seeds (48 runs, resumable) | train + validation | about 2.5 to 4 h |
| 3 | Multi-seed summary, ablation tables, pick the final deep model | validation | under 1 min |
| 4 | Hybrid models: CNN embedding + HOG fusion SVM, confidence-gated cascade | train + validation | about 20 to 30 min |
| 5 | **Freeze, then the one-shot test gate** | test (once) | about 5 min |
| 6 | Analysis: CIs, McNemar, calibration, robustness, inference cost | gate outputs | about 5 min |
| 7 | Download `phase2_results.zip` | | |

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Run the cells from top to bottom.
If Colab disconnects, run Step 1 again and continue: finished training runs are skipped automatically
(they are kept on Google Drive when `USE_DRIVE = True`).

## Step 1. Setup

In [ ]:
# Get the project code. Option A: upload the project zip. Option B: clone your GitHub repository.
import os, subprocess, sys, zipfile

REPO_URL = ""   # e.g. "https://github.com/<user>/fashion-tagging.git" (leave empty to upload the zip instead)
PROJECT_DIR = "/content/fashion-tagging-main"

if not os.path.isdir(PROJECT_DIR):
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    else:
        from google.colab import files
        uploaded = files.upload()                      # choose the project zip
        name = next(iter(uploaded))
        with zipfile.ZipFile(name) as z:
            z.extractall("/content")
        assert os.path.isdir(PROJECT_DIR), "the zip must contain the folder fashion-tagging-main/"
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print("working directory:", os.getcwd())

In [ ]:
# Keep results on Google Drive so that a disconnected runtime does not lose finished training runs.
USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASHION_P2_RESULTS"] = "/content/drive/MyDrive/fashion_phase2/results_phase2"
        os.environ["FASHION_P2_FIGURES"] = "/content/drive/MyDrive/fashion_phase2/figures_phase2"
        os.makedirs(os.environ["FASHION_P2_RESULTS"], exist_ok=True)
        print("results will be kept in", os.environ["FASHION_P2_RESULTS"])
    except Exception as err:
        print("Google Drive not mounted (", err, "): results stay on the Colab disk")

In [ ]:
import torch, sklearn, skimage, numpy, pandas
print("torch", torch.__version__, "| sklearn", sklearn.__version__, "| skimage", skimage.__version__)
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Set Runtime > Change runtime type > T4 GPU, otherwise the full study takes days.")

In [ ]:
# Unit tests of the Phase 2 modules (about 1 minute).
!python -m pytest tests/test_dl_models.py tests/test_augment.py tests/test_dl_train.py tests/test_ablation.py tests/test_hybrid.py tests/test_dl_evaluation.py tests/test_robustness.py tests/test_phase2_experiments.py tests/test_phase2_leakage.py -q

In [ ]:
# Smoke run: every stage on tiny subsets (2 epochs). It writes to results/phase2_smoke and never touches the
# real results or the real test-gate flag. If this cell finishes without an error, the full study will run.
!python run_phase2.py --smoke --quiet

## Step 2. Train all configurations (train + validation only)

16 configurations x 3 seeds. One line is printed per finished run, with an estimate of the time left.
You can interrupt and re-run this cell at any time: finished runs are skipped.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd
from IPython.display import Image, display
from src.phase2_context import make_context
from src.phase2_pipeline import (stage_train, stage_summary, stage_hybrid, stage_freeze, stage_test_gate,
                                 stage_analysis)

ctx = make_context()            # full data, seeds (42, 43, 44)
show = lambda name: display(Image(filename=str(ctx.figures_dir / name), width=900))

In [ ]:
records = stage_train(ctx, verbose=False)

## Step 3. Multi-seed summary, ablations and selection of the final deep model (validation only)

In [ ]:
summary = stage_summary(ctx)
cols = ["name", "val_macro_f1_mean", "val_macro_f1_std", "val_accuracy_mean", "gap_mean", "shirt_recall_mean", "n_params"]
display(summary["summary"][cols].round(4))
display(summary["deltas"][["name", "group", "change", "delta_mean", "delta_min", "delta_max", "consistent_sign"]].round(4))
for name in ("architecture_roadmap.png", "ablation_deltas.png", "generalisation_gap.png", "learning_curves.png", "augmentation_examples.png"):
    show(name)

## Step 4. Hybrid models (train + validation only)

* **H0** PCA(HOG) -> SVM (classical control)
* **H1** CNN embedding -> SVM (control: is it the SVM head or the HOG features?)
* **H2** [CNN embedding, HOG] -> SVM (the hybrid)
* **H3** CNN decides when confident, H2 decides otherwise (confidence-gated cascade)

In [ ]:
hybrid = stage_hybrid(ctx)
display(hybrid["validation"].round(4))
show("cascade_threshold.png")

## Step 5. Freeze, then the one-shot test gate

**Stop and check Steps 3 and 4 first.** After the freeze nothing may be tuned, and the gate opens the test set
exactly once for the frozen list of models. Running this cell a second time raises an error by design.

In [ ]:
sha = stage_freeze(ctx)
gate = stage_test_gate(ctx)

## Step 6. Analysis of the gate outputs

In [ ]:
analysis = stage_analysis(ctx)
display(analysis["metrics"].round(4))
display(analysis["mcnemar"])
display(analysis["inference"].round(4))
for name in ("test_summary.png", "per_class_recall_phase2.png", "cm_final_dl.png", "reliability.png", "robustness.png"):
    show(name)

## Step 7. Download the results

In [ ]:
# Everything the report needs: tables, run records, figures, LaTeX tables. Model weights are left out (large).
import zipfile
from pathlib import Path

out = Path("/content/phase2_results.zip")
skip = {"checkpoints", "hybrid_models"}
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for base, arc in ((ctx.results_dir, "results/phase2"), (ctx.figures_dir, "figures/phase2"), (ctx.tables_dir, "report/phase2_tables")):
        for path in sorted(Path(base).rglob("*")):
            if path.is_file() and not (set(path.relative_to(base).parts) & skip):
                z.write(path, f"{arc}/{path.relative_to(base)}")
print(out, f"{out.stat().st_size / 1e6:.1f} MB")
from google.colab import files
files.download(str(out))